In [2]:
import atlite

path = "data/germany-2025.nc"
module : str = "era5"
year : str = "2025"
lon_bounds = (5.5, 15.5)
lat_bounds = (47.0, 55.5)

cutout = atlite.Cutout( path = path,
    module = module,
    x=slice(*lon_bounds),   # longitude bounds for Germany
    y=slice(*lat_bounds),  # latitude bounds for Germany
    time=year)


cutout.prepare()

/Users/saif/Desktop/FlexGridOptimizer/venv/lib/python3.11/site-packages/atlite/cutout.py:156: UserWarning: Arguments module, x, y, time are ignored, since cutout is already built.
  warn(


<Cutout "germany-2025">
 x = 5.50 ⟷ 15.50, dx = 0.25
 y = 47.00 ⟷ 55.50, dy = 0.25
 time = 2025-01-01 ⟷ 2025-12-31, dt = h
 module = era5
 prepared_features = ['height', 'wind', 'influx', 'temperature', 'runoff']

In [12]:
import atlite
import geopandas as gpd
from shapely.geometry import box

# Create simple boxes for 4 zones (approximate)
cutout = atlite.Cutout(path="data/germany-2025.nc")

zones_shapes = gpd.GeoDataFrame({
    'name': ['Nord', 'West', 'Ost', 'Süd'],
    'geometry': [
        box(6, 52, 15, 55),      # Nord: Niedersachsen, Schleswig-Holstein, Hamburg
        box(6, 49, 9, 52),       # West: NRW, Hessen, Rheinland-Pfalz
        box(11, 50, 15, 53),     # Ost: Brandenburg, Sachsen, Thüringen
        box(7, 47, 13, 50)       # Süd: Bayern, Baden-Württemberg
    ]
}, crs='EPSG:4326')

# Set index to name for proper column naming
zones_shapes = zones_shapes.set_index('name')

zones_shapes.to_file('data/germany_4zones.geojson', driver='GeoJSON')

# Calculate with spatial aggregation
wind_cf = cutout.wind(
    turbine='Vestas_V112_3MW',
    shapes=zones_shapes,
    per_unit=True
)

pv_cf = cutout.pv(
    panel='CSi',
    orientation='latitude_optimal',
    shapes=zones_shapes,
    per_unit=True
)

# Convert to pandas DataFrame
wind_df = wind_cf.to_pandas()
pv_df = pv_cf.to_pandas()

# Export
wind_df.to_csv('data/wind_cf_4zones_aggregated.csv')
pv_df.to_csv('data/pv_cf_4zones_aggregated.csv')

print("✓ Wind CF shape:", wind_df.shape)
print("✓ PV CF shape:", pv_df.shape)
print("\nWind CF preview:")
print(wind_df.head())
print("\nPV CF preview:")
print(pv_df.head())

/Users/saif/Desktop/FlexGridOptimizer/venv/lib/python3.11/site-packages/atlite/resource.py:90: FutureWarning: 'add_cutout_windspeed' for wind turbine
power curves will default to True in atlite relase v0.2.15.
  warnings.warn(msg, FutureWarning)


✓ Wind CF shape: (8760, 4)
✓ PV CF shape: (8760, 4)

Wind CF preview:
name                     Nord      West       Ost       Süd
time                                                       
2025-01-01 00:00:00  0.989543  0.749514  0.850072  0.240686
2025-01-01 01:00:00  0.992777  0.779992  0.861112  0.257526
2025-01-01 02:00:00  0.995757  0.798748  0.887177  0.277494
2025-01-01 03:00:00  0.995413  0.819556  0.911713  0.307796
2025-01-01 04:00:00  0.998344  0.822706  0.928523  0.328637

PV CF preview:
name                 Nord  West  Ost  Süd
time                                     
2025-01-01 00:00:00   0.0   0.0  0.0  0.0
2025-01-01 01:00:00   0.0   0.0  0.0  0.0
2025-01-01 02:00:00   0.0   0.0  0.0  0.0
2025-01-01 03:00:00   0.0   0.0  0.0  0.0
2025-01-01 04:00:00   0.0   0.0  0.0  0.0
